In [2]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

gemma_4_good_hackathon_path = kagglehub.competition_download('gemma-4-good-hackathon')

print('Data source import complete.')


# 🚀 Gemma 4 — The Complete Starter Guide | Text, Vision, Agentic & Fine-Tuning

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h3 style="color: white; margin-top: 0;">🏆 Gemma 4 Good Hackathon — Your Launchpad</h3>
<p>Everything you need to go from zero to a working Gemma 4 project — text generation, image understanding, function calling, thinking mode, and QLoRA fine-tuning — all in one notebook.</p>
</div>

**What this notebook covers:**
1. ✅ Model overview & choosing the right variant  
2. ✅ Setup & loading Gemma 4 on Kaggle GPUs  
3. ✅ Text generation (with & without thinking mode)  
4. ✅ Image understanding (multimodal)  
5. ✅ Function calling / tool use (agentic workflows)  
6. ✅ QLoRA fine-tuning for domain adaptation  
7. ✅ Hackathon strategy & submission tips  

> **⬆️ If this notebook helps you, please upvote!** It keeps the notebook visible for the community.

---


## 1. 📖 Gemma 4 Model Family at a Glance

Gemma 4 is Google DeepMind's most capable open model family, built from Gemini 3 research and released under **Apache 2.0**. Here's a quick comparison:

| Variant | Params | Active Params | Context | Modalities | Best For |
|---------|--------|---------------|---------|------------|----------|
| **E2B** | 5.1B (2.3B eff.) | 2.3B | 128K | Text, Image, Audio | Mobile / edge / RPi |
| **E4B** | 8B (4.5B eff.) | 4.5B | 128K | Text, Image, Audio | On-device apps |
| **26B A4B (MoE)** | 25.2B | 3.8B | 256K | Text, Image | Fast inference, latency-sensitive |
| **31B Dense** | 30.7B | 30.7B | 256K | Text, Image | Max quality, fine-tuning base |

### Key capabilities:
- **Thinking mode** — step-by-step reasoning via `<|think|>` token
- **Function calling** — native tool-use for agentic workflows  
- **Variable resolution images** — respects aspect ratio, no squashing
- **Audio** (E2B/E4B only) — speech recognition, translation
- **140+ languages** out of the box

### For this hackathon on Kaggle:
- **T4 GPU (16 GB)**: Use **E2B** or **E4B** (recommended starting point)
- **P100 GPU (16 GB)**: Same as above
- **2× T4 GPUs (32 GB)**: Can try **26B A4B MoE** with 4-bit quantization

> 💡 **Tip**: The 26B MoE only activates 3.8B params at inference — it's nearly as fast as E4B but much smarter.


## 2. ⚙️ Environment Setup

Let's install the latest dependencies. Gemma 4 requires **Transformers ≥ 5.0** (or the latest nightly).


In [ ]:
# Install / upgrade core dependencies
!pip install -U transformers torch accelerate bitsandbytes -q
!pip install -U peft trl datasets -q  # For fine-tuning section
!pip install -U pillow requests -q     # For image demos

In [3]:
import torch
import gc
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

PyTorch version: 2.11.0
CUDA available: False


### Kaggle Secrets Setup

To access Gemma 4 weights, you need a Hugging Face token. Add it as a Kaggle secret named `HF_TOKEN`:

1. Go to **Add-ons → Secrets** in the Kaggle notebook sidebar  
2. Add a secret with key `HF_TOKEN` and your HF token as the value  
3. Make sure to accept the Gemma 4 license on the [model page](https://huggingface.co/google/gemma-4-E4B-it)


In [4]:
# Authenticate with Hugging Face
from huggingface_hub import login

# Method 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Logged in via Kaggle Secrets")
except Exception:
    # Method 2: Manual token (fallback)
    # login(token="your_token_here")
    print("⚠️ Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

⚠️ Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.


## 3. 💬 Text Generation with Gemma 4

Let's start with the bread and butter — loading the model and generating text. We'll use the **E4B instruction-tuned** variant which fits comfortably on a Kaggle T4 GPU.


In [5]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

# Load processor and model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"✅ Model loaded: {MODEL_ID}")
print(f"   Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

✅ Model loaded: google/gemma-4-E2B-it
   Memory used: 0.00 GB


### 3a. Basic Text Generation (Thinking Disabled)

In [9]:
def generate_response(messages, enable_thinking=False, max_tokens=128):
    """Generate a response from the Gemma 4 model."""
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.6,
            top_p=0.95,
            top_k=64,
            do_sample=True,
        )

    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    parsed = processor.parse_response(response)
    return parsed

# Simple conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on health education."},
    {"role": "user", "content": "Tell me a quick joke."},
]

result = generate_response(messages, enable_thinking=False)
print("=== Response ===")
print(result['content'])

=== Response ===
Why don't scientists trust atoms?

Because they make up everything! 😄


### 3b. Thinking Mode (Chain-of-Thought Reasoning)

Gemma 4's killer feature — enable the `<|think|>` token and the model reasons step by step before answering. This is huge for math, coding, and complex analysis.


In [ ]:
# Enable thinking mode for a reasoning-heavy question
messages = [
    {"role": "system", "content": "You are an expert data scientist."},
    {"role": "user", "content": "I have a dataset of 10,000 medical records with 200 features. "
     "30% of the features have >20% missing values, and the target variable (disease diagnosis) "
     "is heavily imbalanced (95% negative, 5% positive). Walk me through your approach."},
]

result = generate_response(messages, enable_thinking=True, max_tokens=2048)
print("=== Thinking + Response ===")
print(result)

### 3c. Multi-Turn Conversation

Gemma 4 handles multi-turn dialogue natively with `system`, `user`, and `assistant` roles.


In [ ]:
# Multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a climate science educator."},
    {"role": "user", "content": "What are the main greenhouse gases?"},
    {"role": "assistant", "content": "The main greenhouse gases are carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and fluorinated gases. CO2 is the most prevalent from burning fossil fuels."},
    {"role": "user", "content": "Which one has the highest global warming potential per molecule?"},
]

result = generate_response(conversation, enable_thinking=False)
print(result)

## 4. 🖼️ Image Understanding (Multimodal)

Gemma 4 can process images with **variable aspect ratios** — no squashing to a fixed square. Let's try it out.


In [ ]:
from transformers import AutoModelForMultimodalLM

# For multimodal tasks, use AutoModelForMultimodalLM
# (If you already loaded with AutoModelForCausalLM, free that memory first)
del model
gc.collect()
torch.cuda.empty_cache()

mm_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("✅ Multimodal model loaded")

In [ ]:
# Image understanding example
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Cat_August_2010-4.jpg/1200px-Cat_August_2010-4.jpg"
            },
            {
                "type": "text",
                "text": "Describe what you see in this image in detail. What breed might this cat be?"
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(mm_model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = mm_model.generate(**inputs, max_new_tokens=512)

response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
parsed = processor.parse_response(response)
print("=== Image Analysis ===")
print(parsed)

### 4a. Document / Chart Understanding

This is incredibly useful for the hackathon — Gemma 4 can parse documents, charts, and handwriting.


In [ ]:
# OCR / Document understanding example
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4e/Handwriting_of_Albert_Einstein_1924.jpg/800px-Handwriting_of_Albert_Einstein_1924.jpg"
            },
            {
                "type": "text",
                "text": "Can you read and transcribe the handwritten text in this image? Also describe the handwriting style."
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, tokenize=True, return_dict=True,
    return_tensors="pt", add_generation_prompt=True,
).to(mm_model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = mm_model.generate(**inputs, max_new_tokens=512)

response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
print(processor.parse_response(response))

## 5. 🛠️ Function Calling (Agentic Workflows)

Gemma 4 supports **native function calling** — this is essential for the hackathon's agentic track. The model can decide which tools to call, pass structured arguments, and chain actions.

Here's a complete example of building a simple health-assistant agent:


In [ ]:
import json

# Define tools that our agent can use
tools = [
    {
        "name": "check_symptoms",
        "description": "Look up potential conditions based on reported symptoms",
        "parameters": {
            "type": "object",
            "properties": {
                "symptoms": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of symptoms reported by the user"
                },
                "severity": {
                    "type": "string",
                    "enum": ["mild", "moderate", "severe"],
                    "description": "Overall severity level"
                }
            },
            "required": ["symptoms"]
        }
    },
    {
        "name": "find_nearest_clinic",
        "description": "Find the nearest medical clinic based on location",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"},
                "specialty": {"type": "string", "description": "Medical specialty needed"}
            },
            "required": ["latitude", "longitude"]
        }
    }
]

# Build the function-calling prompt
# Gemma 4 supports tool definitions in the system prompt
tool_descriptions = json.dumps(tools, indent=2)

messages = [
    {
        "role": "system",
        "content": f"""You are a health triage assistant. You have access to the following tools:

{tool_descriptions}

When a user describes symptoms, use the check_symptoms tool first.
If the situation seems urgent, also use find_nearest_clinic.
Respond with a JSON function call when appropriate, using the format:
```json
{{"tool": "tool_name", "arguments": {{...}}}}
```"""
    },
    {
        "role": "user",
        "content": "I've been having severe headaches for 3 days, along with blurred vision and nausea. I'm located at coordinates 40.7128, -74.0060."
    }
]

result = generate_response(messages, enable_thinking=True, max_tokens=1024)
print("=== Agent Response ===")
print(result)

### 5a. Building a Complete Agent Loop

For a real hackathon project, you'd wrap this in an execution loop:


In [ ]:
# Pseudocode for a full agentic loop
# (This is the architecture pattern — adapt it for your hackathon project)

AGENT_LOOP_EXAMPLE = (
    "def run_agent(user_query, tools, max_iterations=5):\n"
    "    messages = [\n"
    "        {'role': 'system', 'content': build_system_prompt(tools)},\n"
    "        {'role': 'user', 'content': user_query},\n"
    "    ]\n\n"
    "    for i in range(max_iterations):\n"
    "        # 1. Get model response (with thinking enabled)\n"
    "        response = generate_response(messages, enable_thinking=True)\n\n"
    "        # 2. Check if model wants to call a tool\n"
    "        tool_call = parse_tool_call(response)\n\n"
    "        if tool_call:\n"
    "            # 3. Execute the tool\n"
    "            tool_result = execute_tool(tool_call['tool'], tool_call['arguments'])\n\n"
    "            # 4. Feed result back to the model\n"
    "            messages.append({'role': 'assistant', 'content': response})\n"
    "            messages.append({'role': 'user', 'content': f'Tool result: {json.dumps(tool_result)}'})\n"
    "        else:\n"
    "            # 5. No tool call — final answer\n"
    "            return response\n\n"
    "    return 'Max iterations reached.'\n"
)
print(AGENT_LOOP_EXAMPLE)

## 6. 🎯 Fine-Tuning with QLoRA

This is where you differentiate your hackathon project. Fine-tune Gemma 4 on domain-specific data to create a specialized model. QLoRA lets you do this on a single Kaggle T4 GPU!


In [ ]:
from transformers import BitsAndBytesConfig

# Free memory from previous sections
del mm_model
gc.collect()
torch.cuda.empty_cache()

# Quantization config for 4-bit QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load quantized model for fine-tuning
ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model.config.use_cache = False
print("✅ 4-bit quantized model loaded for fine-tuning")
print(f"   Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
ft_model = prepare_model_for_kbit_training(ft_model)

# LoRA configuration — these are good defaults for Gemma 4
lora_config = LoraConfig(
    r=16,                        # Rank — higher = more capacity, more VRAM
    lora_alpha=32,               # Scaling factor
    target_modules=[             # Which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

ft_model = get_peft_model(ft_model, lora_config)

# Show trainable parameters
trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in ft_model.parameters())
print(f"\n📊 Trainable parameters: {trainable_params:,} / {total_params:,}")
print(f"   That's only {100 * trainable_params / total_params:.2f}% of the model!")

### 6a. Prepare Your Training Data

For the hackathon, format your domain-specific data as conversations. Here's an example for a **health education** use case:


In [ ]:
from datasets import Dataset

# Example: Health education Q&A dataset
# Replace this with YOUR domain-specific data for the hackathon!
training_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a community health educator providing accurate, accessible health information."},
            {"role": "user", "content": "What are the early warning signs of dehydration in children?"},
            {"role": "assistant", "content": "Early signs of dehydration in children include: dry mouth and tongue, fewer wet diapers (less than 6 per day for infants), no tears when crying, sunken soft spot on the head (in babies), and unusual sleepiness or fussiness. If you notice these signs, offer small sips of water or an oral rehydration solution frequently. Seek immediate medical attention if the child has not urinated in 8+ hours, has very dry mouth, or seems extremely lethargic."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a community health educator providing accurate, accessible health information."},
            {"role": "user", "content": "How can I prevent mosquito-borne diseases in my community?"},
            {"role": "assistant", "content": "Preventing mosquito-borne diseases involves three key strategies: (1) Eliminate standing water where mosquitoes breed — empty containers, old tires, and flower pot saucers weekly. (2) Personal protection — use insect repellent with DEET or picaridin, wear long sleeves at dawn/dusk, and sleep under insecticide-treated bed nets. (3) Community action — organize neighborhood clean-up drives, support local fumigation programs, and report stagnant water to health authorities."}
        ]
    },
    # Add more examples... For a real project, aim for 100-1000+ examples
]

# Format for SFTTrainer
def format_conversation(example):
    text = processor.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(training_examples)
dataset = dataset.map(format_conversation)
print(f"✅ Dataset ready: {len(dataset)} examples")
print(f"\nSample formatted text (first 500 chars):\n{dataset[0]['text'][:500]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training configuration optimized for Kaggle T4
training_args = TrainingArguments(
    output_dir="./gemma4-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=1,
    save_strategy="epoch",
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=dataset,
)

print("🏋️ Starting fine-tuning...")
print("   (With only 2 examples this is just a demo — use 100+ for real training)")

# Uncomment to actually train:
# trainer.train()
# trainer.save_model("./gemma4-finetuned-final")
# print("✅ Training complete! Model saved.")

### 6b. Saving & Sharing Your Fine-Tuned Model

After training, push your LoRA weights to Hugging Face for your hackathon submission:

```python
# Save adapter weights
trainer.model.save_pretrained("./gemma4-health-adapter")
processor.save_pretrained("./gemma4-health-adapter")

# Push to Hugging Face Hub (for your hackathon submission)
trainer.model.push_to_hub("your-username/gemma4-health-educator")
processor.push_to_hub("your-username/gemma4-health-educator")
```


## 7. 🏆 Hackathon Strategy & Submission Tips

Based on past Kaggle hackathon winners (Gemma 3n Impact Challenge finalists), here's what the judges look for:

### Track Selection Guide

| Track | Best Model | Key Demo Idea |
|-------|-----------|---------------|
| Health & Sciences | 31B or E4B fine-tuned | Medical triage assistant for rural clinics |
| Global Resilience | E2B/E4B (edge) | Offline disaster response coordinator |
| Future of Education | 26B MoE | Adaptive tutoring agent with tool use |
| Digital Equity | E4B multilingual | 140+ language translator for community health |
| Safety & Trust | Any + thinking mode | Explainable AI decision system |

### Submission Checklist

1. **Video (3 min max)** — This is THE most important piece. Tell a story:
   - 30 sec: The problem (make it emotional, real)
   - 60 sec: Your solution demo  
   - 60 sec: Technical architecture & Gemma 4 specifics
   - 30 sec: Impact & scalability

2. **Kaggle Writeup (≤1,500 words)** — Your technical proof:
   - Architecture diagram
   - How you specifically used Gemma 4 (which variant, why)
   - Challenges and solutions
   - Benchmarks / evaluation metrics

3. **Public Code Repository** — Clean, documented, reproducible

4. **Live Demo** — Even a simple Gradio/Streamlit app counts!

### Pro Tips:
- **Edge deployment** scores well — show Gemma 4 running on a Raspberry Pi or phone
- **Domain adaptation** via fine-tuning shows technical depth
- **Agentic workflows** with function calling demonstrate Gemma 4's unique capabilities
- **Multilingual** projects stand out — leverage the 140+ language support
- **Think mode** for explainability — show the model's reasoning chain


### Quick Gradio Demo Template

Here's a template to build a live demo for your submission:

```python
import gradio as gr

def respond(message, history):
    messages = [{"role": "system", "content": "Your system prompt here"}]
    for h in history:
        messages.append({"role": "user", "content": h[0]})
        if h[1]:
            messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})
    
    result = generate_response(messages, enable_thinking=True)
    return result

demo = gr.ChatInterface(
    respond,
    title="🏥 Gemma 4 Health Assistant",
    description="Powered by Gemma 4 E4B — Fine-tuned for community health",
)
demo.launch(share=True)  # Creates a public link for your submission!
```


## 8. 📚 Resources & Next Steps

### Official Links
- [Gemma 4 Model Card (HuggingFace)](https://huggingface.co/collections/google/gemma-4)
- [Gemma 4 Blog Post](https://blog.google/innovation-and-ai/technology/developers-tools/gemma-4/)
- [Gemma 4 Documentation](https://ai.google.dev/gemma/docs/core)
- [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)
- [Google AI Studio (try 31B online)](https://aistudio.google.com/)

### Model Weights
- [Gemma 4 31B Dense](https://huggingface.co/google/gemma-4-31B-it)
- [Gemma 4 26B MoE](https://huggingface.co/google/gemma-4-26B-A4B-it)
- [Gemma 4 E4B](https://huggingface.co/google/gemma-4-E4B-it)
- [Gemma 4 E2B](https://huggingface.co/google/gemma-4-E2B-it)

### Fine-Tuning Resources
- [QLoRA with Gemma (Official Guide)](https://ai.google.dev/gemma/docs/core/huggingface_text_finetune_qlora)
- [Unsloth (2x faster fine-tuning)](https://github.com/unslothai/unsloth)
- [TRL Documentation](https://huggingface.co/docs/trl)

---

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; text-align: center;">
<h3 style="color: white;">⬆️ If this notebook helped you, please upvote! ⬆️</h3>
<p>Good luck with the Gemma 4 Good Hackathon — go build something amazing! 🚀</p>
</div>


In [ ]:
# Cleanup
gc.collect()
torch.cuda.empty_cache()
print("✅ All done! Memory cleaned up.")
print("\n🎯 Next steps:")
print("   1. Fork this notebook")
print("   2. Pick a hackathon track")
print("   3. Replace the example data with your domain data")
print("   4. Fine-tune and build your demo")
print("   5. Submit and win! 🏆")